# Exp 1 — Class Imbalance: WeightedRandomSampler + FocalLoss

**Hypothesis:** WRS + FocalLoss combined should improve minority class recall,
especially Grade 1 (Mild DR) which baseline scores F1=0.026.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))  # so 'from utils import *' works
from utils import *
print("utils loaded ✓")

/scratch/AML_G14_CW/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


utils.py loaded — device=cuda, train=24588, val=5269, test=5269
utils loaded ✓


In [2]:
EXP_DIR  = ROOT / 'outputs' / 'swinv2_small_exp1'
CKPT_DIR_E = EXP_DIR / 'checkpoints'
FIG_DIR_E  = EXP_DIR / 'figures'
for d in (CKPT_DIR_E, FIG_DIR_E): d.mkdir(parents=True, exist_ok=True)
CKPT_PATH  = CKPT_DIR_E / 'exp1_best.pt'

In [3]:
# ── DataLoaders with WeightedRandomSampler ────────────────────────────────────
train_ds    = DRDataset(train_df, TRAIN_DIR, transform=baseline_train_transform)
val_ds      = DRDataset(val_df,   VAL_DIR,   transform=val_transform)
test_ds     = DRDataset(test_df,  TEST_LABELS_DIR, transform=val_transform)
t_loader    = make_loader(train_ds, sampler=wrs_sampler)
v_loader    = make_val_loader(val_ds)
test_loader = make_val_loader(test_ds)
print(f'WeightedRandomSampler active | Train: {len(t_loader)} batches')

print('\nClass weights:')
for name, w in zip(CLASS_NAMES, class_weights_np):
    print(f'  {name:<20}: {w:.4f}')

WeightedRandomSampler active | Train: 1536 batches

Class weights:
  No DR               : 0.2722
  Mild                : 2.8758
  Moderate            : 1.3276
  Severe              : 8.0484
  Proliferative       : 9.9145


In [4]:
# ── Model + FocalLoss ─────────────────────────────────────────────────────────
set_seed()
model     = build_swin_model(pretrained=True).to(device)
criterion = FocalLoss(alpha=class_weights_t, gamma=2.0)
optimizer = layer_wise_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)
print('Loss: FocalLoss (gamma=2.0, alpha=inverse-freq class weights)')

Downloaded and loaded.
SwinV2-Small | 49.0M params
Loss: FocalLoss (gamma=2.0, alpha=inverse-freq class weights)


In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
history, best_kappa = train_loop(
    model, criterion, optimizer, scheduler, scaler,
    t_loader, v_loader, CKPT_PATH)

Epoch       Tr Loss     Tr κ   Val Loss    Val κ
────────────────────────────────────────────────────


1            3.3071   0.5412     1.1511   0.2310  ★


2            2.2261   0.6966     1.0189   0.2759  ★


3            1.5599   0.7531     1.1014   0.2854  ★


4            1.1050   0.7935     1.0920   0.3337  ★


train:  90%|█████████ | 1387/1536 [08:35<00:56,  2.66it/s]

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────────────────
y_true, y_pred = evaluate(model, criterion, optimizer, scaler,
                           test_loader, CKPT_PATH, 'Exp1')
plot_confusion(y_true, y_pred, 'Exp1 WRS+Focal', FIG_DIR_E / 'exp1_confusion.png', cmap='Oranges')

In [ ]:
# ── Compare vs baseline (load baseline results if available) ─────────────────
import json as _json
exp1_qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
exp1_f1  = f1_score(y_true, y_pred, average=None).tolist()

results = {'qwk': exp1_qwk, 'macro_f1': f1_score(y_true, y_pred, average='macro'),
           'f1_per': exp1_f1, 'history': history}
with open(EXP_DIR / 'exp1_results.json', 'w') as f:
    _json.dump(results, f, indent=2)

# Try loading baseline for delta comparison
try:
    with open(ROOT/'outputs'/'swinv2_small_baseline'/'baseline_results.json') as f:
        b = _json.load(f)
    print('\n' + '='*60)
    print(f'  {"Metric":<20} {"Baseline":>10}  {"Exp1":>8}  {"Delta":>8}')
    print('='*60)
    print(f'  {"QWK":<20} {b["qwk"]:>10.4f}  {exp1_qwk:>8.4f}  {exp1_qwk-b["qwk"]:>+8.4f}')
    print(f'\n  Per-class F1:')
    for name, fb, fe in zip(CLASS_NAMES, b['f1_per'], exp1_f1):
        print(f'    {name:<20}: {fb:.4f}  {fe:.4f}  ({fe-fb:+.4f})')
except FileNotFoundError:
    print(f'Exp1 QWK: {exp1_qwk:.4f}')
    print('(Run baseline notebook first to get comparison table)')